In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

C:\Users\User\AppData\Local\Temp\ipykernel_10252\2553943042.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
import sys
!{sys.executable} -m pip install langchain langchain-community langchain-text-splitters langchain-chroma langchain-google-genai langchain-groq langgraph faiss-cpu pypdf python-dotenv langchain-core


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
loader = PyPDFLoader("ai.pdf")
documents = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(documents)
print(f"Total chunks: {len(chunks)}")
print(chunks[0].page_content[:300])

Total chunks: 4
Artificial Intelligence (AI) simulates human intelligence through machines, while Machine 
Learning (ML) is a subset of AI that enables systems to learn from data without explicit 
programming. ML primarily operates through Supervised Learning (labeled data), 
Unsupervised Learning (pattern discover


In [4]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = Chroma.from_documents(
    documents=chunks, embedding=embeddings, persist_directory="./chroma_db"
)
print("Database Created!")

Database Created!


In [5]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke("What is Machine Learning?")
for doc in docs:
    print(doc.page_content)
    print("---")

Artificial Intelligence (AI) simulates human intelligence through machines, while Machine 
Learning (ML) is a subset of AI that enables systems to learn from data without explicit 
programming. ML primarily operates through Supervised Learning (labeled data), 
Unsupervised Learning (pattern discovery), and Reinforcement Learning (trial and 
error).  
Key Concepts & Algorithms 
1. Supervised Learning 
Models are trained on known input-output pairs to predict future outcomes.
---
Artificial Intelligence (AI) simulates human intelligence through machines, while Machine 
Learning (ML) is a subset of AI that enables systems to learn from data without explicit 
programming. ML primarily operates through Supervised Learning (labeled data), 
Unsupervised Learning (pattern discovery), and Reinforcement Learning (trial and 
error).  
Key Concepts & Algorithms 
1. Supervised Learning 
Models are trained on known input-output pairs to predict future outcomes.
---
Models are trained on known input-

In [6]:
import glob
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

DOCS_DIR = "sample_docs"
INDEX_DIR = "faiss_index"

def load_and_split(docs_dir):
    chunks = []
    for path in glob.glob(os.path.join(docs_dir, "*.txt")):
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        source = os.path.basename(path)
        paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
        current = ""
        for para in paragraphs:
            if len(current) + len(para) + 2 <= 500:
                current = (current + "\n\n" + para).strip()
            else:
                if current:
                    chunks.append(Document(page_content=current, metadata={"source": source}))
                current = para
        if current:
            chunks.append(Document(page_content=current, metadata={"source": source}))
    return chunks

rag_chunks = load_and_split(DOCS_DIR)
print(f"{len(rag_chunks)} chunks")

vectorstore2 = FAISS.from_documents(rag_chunks, embeddings)
vectorstore2.save_local(INDEX_DIR)
print(f"Index saved to {INDEX_DIR}/")

2 chunks
Index saved to faiss_index/


In [7]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Adds a and b."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiplies a and b."""
    return a * b

@tool
def divide(a: int, b: int) -> float:
    """Divides a by b."""
    return a / b

@tool
def search_docs(query: str) -> str:
    """Search the knowledge base for information about AI/ML concepts,
    LangGraph, RAG, embeddings, transformers, and related topics."""
    vs = FAISS.load_local(INDEX_DIR, embeddings, allow_dangerous_deserialization=True)
    docs = vs.as_retriever(search_kwargs={"k": 3}).invoke(query)
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

tools = [add, multiply, divide, search_docs]
tools_by_name = {t.name: t for t in tools}

In [8]:
from langchain_groq import ChatGroq
import operator
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langgraph.graph import StateGraph, END

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
model_with_tools = model.bind_tools(tools)

class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    llm_calls: int

In [9]:
def llm_call(state: MessagesState) -> dict:
    response = model_with_tools.invoke(
        [SystemMessage(content=(
            "You are a helpful assistant that can perform arithmetic "
            "and answer questions about AI/ML concepts. "
            "Use search_docs for AI/ML questions, math tools for calculations."
        ))] + state["messages"]
    )
    return {"messages": [response], "llm_calls": state.get("llm_calls", 0) + 1}

def tool_node(state: MessagesState) -> dict:
    results = []
    for tool_call in state["messages"][-1].tool_calls:
        t = tools_by_name[tool_call["name"]]
        observation = t.invoke(tool_call["args"])
        results.append(ToolMessage(content=str(observation), tool_call_id=tool_call["id"]))
    return {"messages": results}

def should_continue(state: MessagesState) -> Literal["tool_node", "__end__"]:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tool_node"
    return END

In [10]:
graph = StateGraph(MessagesState)
graph.add_node("llm_call", llm_call)
graph.add_node("tool_node", tool_node)
graph.set_entry_point("llm_call")
graph.add_conditional_edges("llm_call", should_continue, {"tool_node": "tool_node", END: END})
graph.add_edge("tool_node", "llm_call")
agent = graph.compile()

In [11]:
result = agent.invoke({"messages": [HumanMessage(content="What is a transformer in AI?")], "llm_calls": 0})
for m in result["messages"]:
    print(f"{m.type}: {m.content}")

human: What is a transformer in AI?
ai: 
tool: Machine learning is a field of AI where systems learn patterns from data without being explicitly programmed.

A transformer is a neural network architecture that uses self-attention to process sequences of data, and it is the foundation of most modern large language models.

---

Retrieval-Augmented Generation (RAG) combines a retrieval system with a language model, so the model can answer questions using external documents instead of relying only on what it memorized during training.

LangGraph is a framework for building stateful, multi-step AI agents as graphs of nodes, where each node can call tools or the LLM.
ai: A **transformer** is a type of neural‑network architecture that was introduced in the 2017 paper *“Attention Is All You Need.”*  
Its key innovation is the **self‑attention mechanism**, which lets the model weigh the importance of every element in a sequence relative to every other element. This allows the network to captur

In [12]:
result2 = agent.invoke({"messages": [HumanMessage(content="What is 25 multiplied by 4?")], "llm_calls": 0})
for m in result2["messages"]:
    print(f"{m.type}: {m.content}")

human: What is 25 multiplied by 4?
ai: 
tool: 100
ai: 25 multiplied by 4 equals **100**.
